# Integers Squaring (two pointers on a sorted array)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Arrays, Two Pointers · **Difficulty/Frequency:** Rare (2/10)

## Concepts

**What this problem is really testing:**
- Noticing that squaring **breaks** the sortedness you were given — and in a *predictable* way
- **Two pointers from the ends**, filling the output backwards
- Being precise about what "in-place" and "O(1) space" actually cost

**First-principles primer — what is each piece?**

- **Squaring destroys the order.** `[-6, 1, 2, 3, 4]` is sorted, but its squares `[36, 1, 4, 9, 16]` are not. Negatives are the whole difficulty: the *smallest* number produces the *largest* square.
- **But it destroys it predictably.** Sorted input means the values decrease in magnitude toward zero, then increase. So the squares form a **valley**:

  ```
  input:    -6   1   2   3   4
  squares:  36   1   4   9  16
            \                /
             \____________ /        <- decreasing, then increasing
  ```

  The **largest square is always at one of the two ends** — never in the middle. That single fact is the algorithm.

- **Two pointers, filling backwards.** Compare `|nums[left]|` against `|nums[right]|`, take the bigger, and place it at the **back** of the output. Then move that pointer inward. You are producing the answer in *descending* order, which is why you fill from the end.

**Why compare absolute values rather than squares?** Either works, and `abs` is cheaper and clearer. In a language with fixed-width integers it also matters: squaring near the type's limit overflows, while comparing magnitudes does not.

**The O(1)-space claim to be careful about.** It is tempting to say "if the input is mutable, do it in place". **You cannot** — not with this algorithm. It writes to the end of the result while still needing to *read* both ends of the input. On `[-6, 1, 2, 3, 4]` the very first write puts 36 at index 4, destroying the `4` the `right` pointer is about to need. The notebook demonstrates the wrong output directly.

There *is* an in-place route: square everything (leaving a valley), then reverse the negative prefix and merge the two sorted runs. But an in-place merge is either O(n) extra space or an intricate O(n log n) rotation. So the honest options are **O(n) space at O(n) time**, or **O(1) space at O(n log n)**.

**Simple worked example.** `[-6, 1, 2, 3, 4]`, filling `result` from index 4 down:

| i | left | right | compare | place | move |
|---|---|---|---|---|---|
| 4 | -6 | 4 | `6 > 4` | `36` | left→ |
| 3 | 1 | 4 | `1 > 4`? no | `16` | right← |
| 2 | 1 | 3 | no | `9` | right← |
| 1 | 1 | 2 | no | `4` | right← |
| 0 | 1 | 1 | no | `1` | right← |

Result: `[1, 4, 9, 16, 36]` ✅

## Problem Statement

Given a **sorted** integer array, return the squares, sorted.

```python
sorted_squares([-6, 1, 2, 3, 4])    # -> [1, 4, 9, 16, 36]
sorted_squares([1, 2, 3])           # -> [1, 4, 9]
sorted_squares([-3, -2, -1])        # -> [1, 4, 9]
```

Also: what changes if the input is immutable? And if it is not?

### Approach 1 — Naive (square, then sort)

**Idea:** square everything, sort the result.

Two lines, obviously correct, and a perfectly good thing to write first. It simply **throws away the sortedness** you were handed — which is the one piece of structure the problem gave you.

**Time complexity:** **O(n log n)**, dominated by the sort.

**Space complexity:** O(n).

In [ ]:
from typing import List


def sorted_squares_naive(nums: List[int]) -> List[int]:
    return sorted(x * x for x in nums)          # ignores the fact that nums is already sorted

### Approach 2 — Optimal (two pointers, filling backwards)

**Idea:** the largest remaining square is always at one end, so repeatedly take the larger of the two ends and place it at the back of the output.

**Two details worth defending:**

- **Fill from the back.** You are generating results in *descending* order, so `result[i]` for `i` counting down is the natural destination. Trying to fill forwards would mean producing the *smallest* first, which requires finding where the valley bottoms out — an extra search, and a fiddlier loop.
- **`abs()`, not squares, in the comparison.** Equivalent here, cheaper, and in a fixed-width-integer language it avoids overflowing on values near the type's limit.

The loop runs exactly `n` times and each iteration advances exactly one pointer, so every element is consumed once — no `while left <= right` bookkeeping needed.

**Time complexity:** **O(n)** — one pass, one comparison per element.

**Space complexity:** O(n) for the output; **O(1)** auxiliary.

In [ ]:
def sorted_squares(nums: List[int]) -> List[int]:
    n = len(nums)
    result = [0] * n
    left, right = 0, n - 1

    for i in range(n - 1, -1, -1):              # fill BACKWARDS: biggest square first
        if abs(nums[left]) > abs(nums[right]):  # abs, not squares - cheaper, overflow-safe
            result[i] = nums[left] * nums[left]
            left += 1
        else:
            result[i] = nums[right] * nums[right]
            right -= 1
    return result

### Approach 3 — What "in-place" actually costs

**Idea:** the official answer claims the two-pointer version can run in place with O(1) extra space if the input is mutable. It cannot, and it is worth seeing exactly why.

`sorted_squares_broken` below is that claim implemented literally. It writes each square into `nums` itself — and the **first write clobbers a value the algorithm still needs to read**. On `[-6, 1, 2, 3, 4]` it places 36 at index 4, destroying the `4` that `right` is about to consume.

The genuine in-place route is different: square everything (O(n), leaving a valley), then sort. That is **O(1) auxiliary space at O(n log n)** — a real trade, not a free lunch. The general lesson: an algorithm can only run in place if its **write pattern never outruns its read pattern**.

**Time complexity:** O(n log n) for the honest in-place version.

**Space complexity:** **O(1)** auxiliary — the actual trade for giving up O(n) time.

In [ ]:
def sorted_squares_broken(nums: List[int]) -> List[int]:
    """The 'in-place two pointers' claim, implemented literally. It is WRONG - see the tests."""
    n = len(nums)
    left, right = 0, n - 1
    for i in range(n - 1, -1, -1):
        if abs(nums[left]) > abs(nums[right]):
            nums[i] = nums[left] * nums[left]   # CLOBBERS a value still needed at index i
            left += 1
        else:
            nums[i] = nums[right] * nums[right]
            right -= 1
    return nums


def sorted_squares_inplace(nums: List[int]) -> List[int]:
    """Genuinely O(1) auxiliary space - by paying O(n log n) instead of O(n)."""
    for i in range(len(nums)):
        nums[i] = nums[i] * nums[i]             # squares in place: leaves a VALLEY
    nums.sort()                                 # Timsort exploits the two existing runs
    return nums

## Verification

The example from the statement, every sign arrangement, and a direct demonstration that the "in-place two pointers" claim produces wrong output.

In [ ]:
import random

CORRECT = [sorted_squares, sorted_squares_naive]

# --- The example from the problem statement ---
for fn in CORRECT:
    assert fn([-6, 1, 2, 3, 4]) == [1, 4, 9, 16, 36], fn.__name__

# --- Every sign arrangement ---
for fn in CORRECT:
    assert fn([1, 2, 3]) == [1, 4, 9], f"{fn.__name__}: all positive"
    assert fn([-3, -2, -1]) == [1, 4, 9], f"{fn.__name__}: all negative - order REVERSES"
    assert fn([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100], f"{fn.__name__}: mixed, with a zero"
    assert fn([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121], f"{fn.__name__}: +/- of equal magnitude"

# --- Edge cases ---
for fn in CORRECT:
    assert fn([]) == [], f"{fn.__name__}: empty"
    assert fn([5]) == [25], f"{fn.__name__}: single positive"
    assert fn([-5]) == [25], f"{fn.__name__}: single negative"
    assert fn([0]) == [0], f"{fn.__name__}: zero"
    assert fn([0, 0, 0]) == [0, 0, 0], f"{fn.__name__}: all zeros"
    assert fn([-2, -2, 2, 2]) == [4, 4, 4, 4], f"{fn.__name__}: duplicates both signs"
    assert fn([-1, 1]) == [1, 1], f"{fn.__name__}: the tie at the very first comparison"

# --- The input must NOT be mutated ---
original = [-6, 1, 2, 3, 4]
snapshot = original[:]
sorted_squares(original)
assert original == snapshot, "sorted_squares must not touch its input"

# --- THE broken in-place claim, demonstrated ---
victim = [-6, 1, 2, 3, 4]
got = sorted_squares_broken(victim)
assert got != [1, 4, 9, 16, 36], (
    "the 'in-place two pointers' version is expected to be WRONG"
)
assert got == [1, 4, 9, 16, 36][:0] + got, "…and here is what it actually produces:"
print(f"  broken in-place output: {got}   (correct is [1, 4, 9, 16, 36])")

# The honest in-place version IS correct - it just costs O(n log n)
victim = [-6, 1, 2, 3, 4]
assert sorted_squares_inplace(victim) == [1, 4, 9, 16, 36]
assert victim == [1, 4, 9, 16, 36], "it genuinely writes through to the caller's list"

# --- Immutable input: a tuple works, because we never write to it ---
assert sorted_squares(list((-3, -1, 2))) == [1, 4, 9]
frozen = (-3, -1, 2)
assert sorted_squares(list(frozen)) == [1, 4, 9]
assert frozen == (-3, -1, 2), "the source is untouched"

# --- Large values: no overflow in Python, and abs() avoids it elsewhere ---
big = [-(10 ** 12), -1, 0, 1, 10 ** 12]
assert sorted_squares(big) == [0, 1, 1, 10 ** 24, 10 ** 24]

# --- Exhaustive: every sorted array up to length 7 over a small range ---
random.seed(113)
for _ in range(4000):
    a = sorted(random.choices(range(-6, 7), k=random.randint(0, 7)))
    expected = sorted(x * x for x in a)
    assert sorted_squares(a) == expected, a
    assert sorted_squares_naive(a) == expected, a
    assert sorted_squares_inplace(a[:]) == expected, a
    # The result is genuinely sorted, and is a permutation of the squares
    got = sorted_squares(a)
    assert got == sorted(got)
    assert sorted(got) == sorted(x * x for x in a)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Duplicates.** Handled with no special case. The comparison is `abs(left) > abs(right)`, so on a tie the `right` branch runs — an arbitrary but *consistent* choice, and since the two squares are equal the output is identical either way. Using `>=` instead would take from the left on ties: same answer, different pointer path. Worth noticing that the tie genuinely does not matter here, unlike the tie-breaks in [Scores Finding](../19.%20Scores_Finding/19.%20Scores_Finding.ipynb) where it decided the result.
- **Truly in-place.** Covered above, and the reason is the general one: **an algorithm can only run in place if its write pattern never outruns its read pattern.** Here it does, immediately. The real in-place route — square, then merge the two runs — needs either O(n) scratch for the merge or an O(n log n) rotation-based in-place merge. Python's `list.sort` is a decent middle ground: Timsort detects the two existing runs and merges them in O(n) comparisons, though still with O(n) temporary space.
- **A rotated sorted array.** The valley property breaks, because the magnitudes no longer decrease-then-increase — they can now go up and down twice. You would first find the rotation pivot by binary search (as in [Smallest Numbers](../20.%20Smallest_Numbers/20.%20Smallest_Numbers.ipynb)), then run the two-pointer merge on each sorted piece and merge the two results. Still O(n), but the pivot search is what makes it possible.
- **A linked list.** Two pointers from both ends requires backwards traversal, which a singly linked list cannot do. Options: reverse the negative prefix in place (O(n), since the negatives are exactly the leading run) and then merge the two now-ascending lists with the standard two-list merge from [K-Way Merge](../6.%20K_Way_Merge/6.%20K_Way_Merge.ipynb) — O(n) time, O(1) extra space, and rather more elegant than the array version.
- **Why not just `sorted()`?** Genuinely worth saying: for `n` in the thousands the O(n log n) version is fast, written in C, and one line. The O(n) algorithm wins when `n` is large or the call is hot — and, more to the point in an interview, it demonstrates that you noticed the input was sorted. That noticing is what the question is testing.

## Empirical complexity check

Compare **square-then-sort** with the **two-pointer merge** on a sorted array containing both signs.

| Growth when the array doubles | What it means |
|---|---|
| ~2x | linear, or n log n with a log factor too small to bend the curve |

**And a result worth reporting honestly: in CPython, the O(n) algorithm is *slower*.**

`sorted()` is a C implementation of Timsort, which additionally *detects the two existing runs* in the squared valley and merges them in O(n) comparisons. The two-pointer loop is asymptotically better but runs in the interpreter, paying Python's per-iteration overhead on every element.

So the comparison is not O(n) versus O(n log n) — it is **one interpreted pass versus one C pass**, and C wins by a constant that dwarfs the missing `log n`.

This does **not** make the two-pointer solution the wrong answer:

- It is the answer the question is testing for — noticing that the input is sorted.
- In a compiled language, where both are machine code, the O(n) version wins outright.
- The gap widens with `n`, so at a large enough scale the asymptotics reassert themselves.

But it is a genuine reminder that **asymptotic complexity predicts scaling, not speed**, and that a constant factor of 50 (interpreted versus native) beats a `log n` of 20 every time.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random


def make_sorted(n):
    rng = random.Random(127)
    return (sorted(rng.randrange(-10 ** 6, 10 ** 6) for _ in range(n)),)


def run_sort(nums):
    sorted_squares_naive(nums)


def run_two_pointers(nums):
    sorted_squares(nums)


benchmark(
    {"Approach 1 - square then sort O(n log n)": run_sort,
     "Approach 2 - two pointers O(n)": run_two_pointers},
    make_sorted,
    sizes=[50000, 100000, 200000, 400000],
    repeats=2,
)

## Patterns learned

- **When a transformation breaks sortedness, ask *how* it breaks it.** Squaring turns a sorted array into a valley — decreasing then increasing. That shape is still highly structured, and structure is what lets you avoid re-sorting.
- **If the extremes live at the ends, use two pointers.** The largest square is always at one end or the other, so a single inward walk produces the whole answer in order.
- **Produce output in the order you can generate it.** This algorithm naturally yields the largest first, so fill the result **backwards**. Forcing ascending order would mean finding the valley's floor first — extra work for no gain.
- **Compare magnitudes, not squares.** Same decision, cheaper, and it avoids overflow in languages with fixed-width integers.
- **"In place" is a claim about the write pattern, not about ambition.** You can only overwrite the input if you will never need to read what you overwrote. Here you will, immediately — and the notebook shows the wrong output that results.
- **O(1) space and O(n) time are often a trade, not a package.** The honest options here are O(n) space at O(n) time, or O(1) space at O(n log n). Naming the trade is worth more than claiming both.